# Superstore Sales Analysis
**Dataset:** Kaggle Superstore Sales | **Tool:** Python, Pandas, Matplotlib, Seaborn


## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")


## Step 2: Load the Dataset

In [ ]:
# Read the CSV file
df = pd.read_csv('superstore_sales.csv')

# Convert date columns to date format
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print("Dataset loaded successfully!")
print("Shape:", df.shape)


## Step 3: First Look at the Data

In [ ]:
# See first 5 rows
df.head()


In [ ]:
# See column names and data types
df.info()


In [ ]:
# Basic statistics
df.describe()


## Step 4: Data Cleaning

In [ ]:
# Check for missing values
print("Missing values in each column:")
print(df.isnull().sum())


In [ ]:
# Check for duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())


In [ ]:
# Drop duplicates if any
df = df.drop_duplicates()
print("Duplicates removed. New shape:", df.shape)


## Step 5: Add Useful Columns

In [ ]:
# Add Year, Month, Quarter columns from Order Date
df['Year']    = df['Order Date'].dt.year
df['Month']   = df['Order Date'].dt.month
df['Quarter'] = df['Order Date'].dt.quarter

# How many days it took to ship
df['Ship Lag'] = (df['Ship Date'] - df['Order Date']).dt.days

# Profit margin percentage
df['Profit Margin'] = (df['Profit'] / df['Sales'] * 100).round(2)

print("New columns added: Year, Month, Quarter, Ship Lag, Profit Margin")
df[['Sales', 'Profit', 'Ship Lag', 'Profit Margin']].head()


## Step 6: Business KPIs (Key Numbers)

In [ ]:
total_sales    = df['Sales'].sum()
total_profit   = df['Profit'].sum()
avg_margin     = df['Profit Margin'].mean()
total_orders   = df['Order ID'].nunique()
total_customers= df['Customer ID'].nunique()

print("===== BUSINESS KPI SUMMARY =====")
print(f"Total Sales      : ${total_sales:,.0f}")
print(f"Total Profit     : ${total_profit:,.0f}")
print(f"Avg Profit Margin: {avg_margin:.1f}%")
print(f"Total Orders     : {total_orders:,}")
print(f"Total Customers  : {total_customers:,}")
print("================================")


## Step 7: Sales and Profit Distribution

In [ ]:
# Plot histogram of Sales and Profit side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Sales'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Sales Distribution')
axes[0].set_xlabel('Sales ($)')
axes[0].set_ylabel('Count')

axes[1].hist(df['Profit'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Profit Distribution')
axes[1].set_xlabel('Profit ($)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()


## Step 8: Sales by Category

In [ ]:
# Total sales for each category
category_sales = df.groupby('Category')['Sales'].sum().sort_values()

plt.figure(figsize=(8, 5))
plt.barh(category_sales.index, category_sales.values, color=['#1B2A4A','#3D5A80','#4ECDC4'])
plt.title('Total Sales by Category')
plt.xlabel('Total Sales ($)')
plt.tight_layout()
plt.show()


In [ ]:
# Total sales and profit for each sub-category
sub_sales = df.groupby('Sub-Category')['Sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(sub_sales.index, sub_sales.values, color='steelblue', edgecolor='white')
plt.title('Total Sales by Sub-Category')
plt.xlabel('Sub-Category')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## Step 9: Profit Margin by Sub-Category

In [ ]:
# Average profit margin per sub-category
sub_margin = df.groupby('Sub-Category')['Profit Margin'].mean().sort_values()

# Color: red if negative, green if positive
colors = ['red' if x < 0 else 'green' for x in sub_margin.values]

plt.figure(figsize=(12, 6))
plt.barh(sub_margin.index, sub_margin.values, color=colors)
plt.axvline(0, color='black', linewidth=1)
plt.title('Average Profit Margin (%) by Sub-Category')
plt.xlabel('Profit Margin (%)')
plt.tight_layout()
plt.show()


## Step 10: Sales by Customer Segment

In [ ]:
# Sales by segment
segment_sales = df.groupby('Segment')['Sales'].sum()

plt.figure(figsize=(7, 7))
plt.pie(segment_sales.values, labels=segment_sales.index,
        autopct='%1.1f%%', colors=['#1B2A4A','#4ECDC4','#FF6B6B'],
        startangle=90)
plt.title('Sales Share by Customer Segment')
plt.tight_layout()
plt.show()

print(segment_sales)


## Step 11: Monthly Sales Trend

In [ ]:
# Group by year and month
monthly = df.groupby(['Year', 'Month'])['Sales'].sum().reset_index()
monthly['Period'] = pd.to_datetime(
    monthly['Year'].astype(str) + '-' + monthly['Month'].astype(str).str.zfill(2))
monthly = monthly.sort_values('Period')

plt.figure(figsize=(14, 5))
plt.plot(monthly['Period'], monthly['Sales'], color='#1B2A4A', linewidth=2, marker='o', markersize=3)
plt.fill_between(monthly['Period'], monthly['Sales'], alpha=0.1, color='#1B2A4A')
plt.title('Monthly Sales Trend (2019–2022)')
plt.xlabel('Month')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Year-over-year total sales
yearly = df.groupby('Year')['Sales'].sum()

plt.figure(figsize=(8, 5))
plt.bar(yearly.index.astype(str), yearly.values, color=['#1B2A4A','#3D5A80','#4ECDC4','#FF6B6B'], edgecolor='white')
plt.title('Total Sales by Year')
plt.xlabel('Year')
plt.ylabel('Total Sales ($)')
for i, v in enumerate(yearly.values):
    plt.text(i, v + 500, f'${v:,.0f}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Sales by quarter
quarterly = df.groupby(['Year','Quarter'])['Sales'].sum().reset_index()
quarterly['Label'] = 'Q' + quarterly['Quarter'].astype(str) + '-' + quarterly['Year'].astype(str)

plt.figure(figsize=(14, 5))
plt.bar(quarterly['Label'], quarterly['Sales'], color='#3D5A80', edgecolor='white')
plt.title('Quarterly Sales')
plt.xlabel('Quarter')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Step 12: Geographic Analysis

In [ ]:
# Sales by region
region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(region_sales.index, region_sales.values, color=['#1B2A4A','#3D5A80','#4ECDC4','#FF6B6B'], edgecolor='white')
plt.title('Total Sales by Region')
plt.xlabel('Region')
plt.ylabel('Total Sales ($)')
for i, v in enumerate(region_sales.values):
    plt.text(i, v + 200, f'${v:,.0f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Top 10 cities by sales
top_cities = df.groupby('City')['Sales'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 5))
plt.bar(top_cities.index, top_cities.values, color='steelblue', edgecolor='white')
plt.title('Top 10 Cities by Sales')
plt.xlabel('City')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


## Step 13: Correlation Heatmap

In [ ]:
# Select only number columns
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Ship Lag', 'Profit Margin']

# Calculate correlation
corr = df[num_cols].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Between Numeric Columns')
plt.tight_layout()
plt.show()


## Step 14: Discount vs Profit

In [ ]:
# Average profit for each discount level
disc_profit = df.groupby('Discount')['Profit'].mean().reset_index()

plt.figure(figsize=(10, 5))
plt.bar(disc_profit['Discount'].astype(str), disc_profit['Profit'],
        color=['green' if v > 0 else 'red' for v in disc_profit['Profit']],
        edgecolor='white')
plt.axhline(0, color='black', linewidth=1)
plt.title('Average Profit by Discount Level')
plt.xlabel('Discount')
plt.ylabel('Average Profit ($)')
plt.tight_layout()
plt.show()

print("Key Finding: High discounts (>20%) usually result in low or negative profit!")


## Step 15: Shipping Mode Analysis

In [ ]:
# Orders and average ship lag per shipping mode
ship_stats = df.groupby('Ship Mode').agg(
    Total_Orders = ('Order ID', 'count'),
    Avg_Ship_Days= ('Ship Lag', 'mean')
).reset_index()

print(ship_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(ship_stats['Ship Mode'], ship_stats['Total_Orders'],
            color=['#1B2A4A','#3D5A80','#4ECDC4','#FF6B6B'], edgecolor='white')
axes[0].set_title('Number of Orders by Ship Mode')
axes[0].set_ylabel('Orders')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(ship_stats['Ship Mode'], ship_stats['Avg_Ship_Days'],
            color=['#1B2A4A','#3D5A80','#4ECDC4','#FF6B6B'], edgecolor='white')
axes[1].set_title('Avg Shipping Days by Ship Mode')
axes[1].set_ylabel('Days')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()


## Step 16: Customer RFM Segmentation
> RFM = **R**ecency (how recently), **F**requency (how often), **M**onetary (how much)

In [ ]:
# Reference date = day after last order
last_date = df['Order Date'].max() + pd.Timedelta(days=1)

# Calculate R, F, M for each customer
rfm = df.groupby('Customer ID').agg(
    Recency   = ('Order Date',  lambda x: (last_date - x.max()).days),
    Frequency = ('Order ID',    'nunique'),
    Monetary  = ('Sales',       'sum')
).reset_index()

print("RFM Table (first 5 rows):")
print(rfm.head())


In [ ]:
# Score each column 1 to 5 (5 = best customer)
rfm['R_Score'] = pd.qcut(rfm['Recency'],  5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5]).astype(int)

# Total RFM score
rfm['RFM_Total'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

# Label customers based on score
def label_customer(score):
    if score >= 13:
        return 'Champions'
    elif score >= 10:
        return 'Loyal'
    elif score >= 7:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm['RFM_Total'].apply(label_customer)

print("Customer Segment Counts:")
print(rfm['Segment'].value_counts())


In [ ]:
# Pie chart of customer segments
seg_counts = rfm['Segment'].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(seg_counts.values, labels=seg_counts.index,
        autopct='%1.1f%%', colors=['#1B2A4A','#4ECDC4','#FF6B6B','#FFE66D'],
        startangle=140)
plt.title('Customer Segments (RFM)')
plt.tight_layout()
plt.show()


## Step 17: Summary of Insights

| # | Finding |
|---|---------|
| 1 | Technology has the highest total sales |
| 2 | Tables sub-category has the lowest (negative) profit margin |
| 3 | Q4 (October–December) is always the best sales quarter |
| 4 | Discounts above 20% usually result in negative profit |
| 5 | West and East regions generate the most revenue |
| 6 | Champions customers bring the most value — focus on retaining them |
| 7 | Standard Class is the most used shipping mode |


In [ ]:
print("Analysis Complete!")
print(f"Total Revenue : ${df['Sales'].sum():,.0f}")
print(f"Total Profit  : ${df['Profit'].sum():,.0f}")
print(f"Best Region   : {df.groupby('Region')['Sales'].sum().idxmax()}")
print(f"Best Category : {df.groupby('Category')['Sales'].sum().idxmax()}")
print(f"Worst Margin  : {df.groupby('Sub-Category')['Profit Margin'].mean().idxmin()} sub-category")
